# MyDigitalTwin — Amazon
**Notebook 02 — Ingestion, exploration, nettoyage → Parquet**

Source : `data/raw/AMAZON/Your Orders/Your Amazon Orders/Order History.csv`  
Output : `data/parquet/amazon_orders.parquet`

## Objectif ML
- **ALS (axe 2)** : interactions produit → catégorie avec poids d'achat
- **K-Means (axe 3)** : clustering comportemental (montants, catégories, temporalité)

## 0. Initialisation Spark

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.path.dirname('__file__'), '../../..')))
from config import RAW_DATA, WAREHOUSE

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder \
    .appName("MyDigitalTwin - Amazon") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print(f"Spark version : {spark.version}")

Spark version : 3.5.5


26/03/28 12:06:54 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


## 1. Ingestion

In [2]:
RAW_PATH    = os.path.join(RAW_DATA, "AMAZON", "Your Orders", "Your Amazon Orders", "Order History.csv")
PARQUET_OUT = os.path.join(os.path.dirname(os.path.abspath("__file__")), "..", "..", "data", "parquet", "amazon_orders.parquet")

df_raw = spark.read \
    .option("header", "true") \
    .option("encoding", "UTF-8") \
    .option("multiLine", "true") \
    .option("escape", '"') \
    .csv(RAW_PATH)

print(f"Lignes brutes : {df_raw.count():,}")
df_raw.printSchema()

Lignes brutes : 74
root
 |-- ASIN: string (nullable = true)
 |-- Billing Address: string (nullable = true)
 |-- Carrier Name & Tracking Number: string (nullable = true)
 |-- Currency: string (nullable = true)
 |-- Gift Message: string (nullable = true)
 |-- Gift Recipient Contact: string (nullable = true)
 |-- Gift Sender Name: string (nullable = true)
 |-- Item Serial Number: string (nullable = true)
 |-- Order Date: string (nullable = true)
 |-- Order ID: string (nullable = true)
 |-- Order Status: string (nullable = true)
 |-- Original Quantity: string (nullable = true)
 |-- Payment Method Type: string (nullable = true)
 |-- Product Condition: string (nullable = true)
 |-- Product Name: string (nullable = true)
 |-- Purchase Order Number: string (nullable = true)
 |-- Ship Date: string (nullable = true)
 |-- Shipment Item Subtotal: string (nullable = true)
 |-- Shipment Item Subtotal Tax: string (nullable = true)
 |-- Shipment Status: string (nullable = true)
 |-- Shipping Address: 

## 2. Exploration — Data Quality

In [3]:
# Valeurs nulles / Not Available par colonne
print("=== Valeurs manquantes ===")
df_raw.select([
    F.count(
        F.when(F.col(c).isNull() | (F.col(c) == "Not Available") | (F.col(c) == ""), c)
    ).alias(c)
    for c in df_raw.columns
]).show(vertical=True)

# Période couverte
print("\n=== Période ===")
df_raw.agg(
    F.min("Order Date").alias("premiere_commande"),
    F.max("Order Date").alias("derniere_commande")
).show(truncate=False)

# Statuts des commandes
print("\n=== Statuts ===")
df_raw.groupBy("Order Status").count().orderBy(F.desc("count")).show()

=== Valeurs manquantes ===


26/03/28 12:07:06 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


-RECORD 0-----------------------------
 ASIN                           | 0   
 Billing Address                | 0   
 Carrier Name & Tracking Number | 9   
 Currency                       | 0   
 Gift Message                   | 74  
 Gift Recipient Contact         | 74  
 Gift Sender Name               | 74  
 Item Serial Number             | 73  
 Order Date                     | 0   
 Order ID                       | 0   
 Order Status                   | 0   
 Original Quantity              | 0   
 Payment Method Type            | 41  
 Product Condition              | 0   
 Product Name                   | 0   
 Purchase Order Number          | 0   
 Ship Date                      | 9   
 Shipment Item Subtotal         | 9   
 Shipment Item Subtotal Tax     | 9   
 Shipment Status                | 9   
 Shipping Address               | 1   
 Shipping Charge                | 0   
 Shipping Option                | 1   
 Total Amount                   | 0   
 Total Discounts         

In [4]:
# Aperçu des produits et montants
df_raw.select("Product Name", "Total Amount", "Unit Price", "Order Date") \
    .show(10, truncate=60)

+------------------------------------------------------------+------------+----------+--------------------+
|                                                Product Name|Total Amount|Unit Price|          Order Date|
+------------------------------------------------------------+------------+----------+--------------------+
|RAVIAD Câble USB C vers Lightning 3M, [Certifié MFi] Câbl...|        9.97|      9.16|2022-12-21T19:59:54Z|
|UGREEN Câble Lightning vers USB C MFi Certifié Nylon Tres...|        21.1|     12.49|2025-07-07T18:55:42Z|
|RISEOFLE 180cm Trepied Smartphone & Perche a Selfie, Trep...|       14.44|     11.93|2025-12-21T14:24:14Z|
|TONEOF Lampe de Téléphone Améliorée avec Coque Souple et ...|       18.99|     15.69|2025-12-21T14:24:14Z|
|Crucial RAM CT16G4SFD824A 16Go DDR4 2400MHz CL17 Mémoire ...|       78.78|     65.11|2022-04-24T08:57:44Z|
|Spray au sel de mer Spray au sel Eau salée Spray pour che...|       18.28|     16.66|2024-10-02T18:33:46Z|
|JBL Live 660NC - Casque aud

## 3. Nettoyage

Colonnes supprimées (confidentialité) :
- `Billing Address` / `Shipping Address` → adresse postale réelle
- `Carrier Name & Tracking Number` → inutile analytiquement
- `Gift Message` / `Gift Recipient Contact` / `Gift Sender Name` → données tierces
- `Item Serial Number` / `Purchase Order Number` → inutile
- `Shipment Item Subtotal Tax` / `Unit Price Tax` → redondant
- `Product Condition` / `Shipment Status` / `Shipping Option` → peu analytique

In [5]:
# Colonnes à supprimer
DROP_COLS = [
    "Billing Address", "Shipping Address",
    "Carrier Name & Tracking Number",
    "Gift Message", "Gift Recipient Contact", "Gift Sender Name",
    "Item Serial Number", "Purchase Order Number",
    "Shipment Item Subtotal Tax", "Unit Price Tax",
    "Product Condition", "Shipment Status", "Shipping Option",
    "Ship Date"
]

df = df_raw.drop(*DROP_COLS)

print(f"Colonnes restantes : {len(df.columns)}")
df.printSchema()

Colonnes restantes : 14
root
 |-- ASIN: string (nullable = true)
 |-- Currency: string (nullable = true)
 |-- Order Date: string (nullable = true)
 |-- Order ID: string (nullable = true)
 |-- Order Status: string (nullable = true)
 |-- Original Quantity: string (nullable = true)
 |-- Payment Method Type: string (nullable = true)
 |-- Product Name: string (nullable = true)
 |-- Shipment Item Subtotal: string (nullable = true)
 |-- Shipping Charge: string (nullable = true)
 |-- Total Amount: string (nullable = true)
 |-- Total Discounts: string (nullable = true)
 |-- Unit Price: string (nullable = true)
 |-- Website: string (nullable = true)



In [6]:
# Parser la date (format ISO : 2022-12-21T19:59:54Z)
df = df.withColumn(
    "order_date",
    F.to_timestamp(F.col("Order Date"), "yyyy-MM-dd'T'HH:mm:ss'Z'")
)

# Champs temporels
df = df \
    .withColumn("order_year",    F.year("order_date")) \
    .withColumn("order_month",   F.date_format("order_date", "yyyy-MM")) \
    .withColumn("order_weekday", F.dayofweek("order_date")) \
    .withColumn("order_hour",    F.hour("order_date"))

df.select("order_date", "order_year", "order_month", "order_weekday", "order_hour").show(5)

+-------------------+----------+-----------+-------------+----------+
|         order_date|order_year|order_month|order_weekday|order_hour|
+-------------------+----------+-----------+-------------+----------+
|2022-12-21 19:59:54|      2022|    2022-12|            4|        19|
|2025-07-07 18:55:42|      2025|    2025-07|            2|        18|
|2025-12-21 14:24:14|      2025|    2025-12|            1|        14|
|2025-12-21 14:24:14|      2025|    2025-12|            1|        14|
|2022-04-24 08:57:44|      2022|    2022-04|            1|         8|
+-------------------+----------+-----------+-------------+----------+
only showing top 5 rows



In [7]:
# Nettoyer les montants (Total Amount, Unit Price, Shipping Charge, Total Discounts)
# Certains ont un apostrophe en préfixe ex: '-1.11'
for col_name in ["Total Amount", "Unit Price", "Shipment Item Subtotal", "Shipping Charge", "Total Discounts"]:
    df = df.withColumn(
        col_name,
        F.regexp_replace(F.col(col_name), "[^0-9.\\-]", "").cast("double")
    )

# Quantité en entier
df = df.withColumn("Original Quantity", F.col("Original Quantity").cast("int"))

df.select("Product Name", "Total Amount", "Unit Price", "Shipping Charge", "Total Discounts").show(5, truncate=50)

+--------------------------------------------------+------------+----------+---------------+---------------+
|                                      Product Name|Total Amount|Unit Price|Shipping Charge|Total Discounts|
+--------------------------------------------------+------------+----------+---------------+---------------+
|RAVIAD Câble USB C vers Lightning 3M, [Certifié...|        9.97|      9.16|            0.0|          -1.11|
|UGREEN Câble Lightning vers USB C MFi Certifié ...|        21.1|     12.49|           5.99|            0.0|
|RISEOFLE 180cm Trepied Smartphone & Perche a Se...|       14.44|     11.93|           0.26|            0.0|
|TONEOF Lampe de Téléphone Améliorée avec Coque ...|       18.99|     15.69|           0.26|            0.0|
|Crucial RAM CT16G4SFD824A 16Go DDR4 2400MHz CL1...|       78.78|     65.11|            0.0|            0.0|
+--------------------------------------------------+------------+----------+---------------+---------------+
only showing top 5 

In [8]:
# Catégorisation automatique des produits par mots-clés
def categorize_product(col):
    name = F.lower(col)
    return (
        F.when(name.rlike(r"cable|usb|chargeur|batterie|ecouteur|casque|clavier|souris|hub|adaptateur|hdmi|ssd|led|lampe|trepied|camera|webcam|microphone|smartphone"), "Électronique")
         .when(name.rlike(r"t-shirt|chemise|pantalon|jean|veste|chaussure|sneaker|pull|sweat|short|chaussette"), "Vêtements")
         .when(name.rlike(r"livre|book|roman|guide|manuel"), "Livres")
         .when(name.rlike(r"sport|fitness|yoga|musculation|velo|running|basketball|football|tennis"), "Sport")
         .when(name.rlike(r"dj|vinyle|platine|mixeur|controller|cdj|pioneer|rekordbox|studio"), "Musique/DJ")
         .when(name.rlike(r"jeu|game|gaming|manette|console|playstation|xbox|nintendo|switch"), "Jeux vidéo")
         .when(name.rlike(r"cuisine|casserole|poele|couteau|mixeur|cafe|the"), "Cuisine")
         .when(name.rlike(r"shampoing|creme|soin|vitamine|rasoir|parfum|deodorant"), "Beauté/Santé")
         .when(name.rlike(r"prime|abonnement|membership|subscription"), "Abonnement")
         .when(name.rlike(r"lampe|cadre|coussin|etagere|rangement|nettoyage"), "Maison")
         .otherwise("Autre")
    )

df = df.withColumn("category", categorize_product(F.col("Product Name")))

print("=== Répartition par catégorie ===")
df.groupBy("category").count().orderBy(F.desc("count")).show()

=== Répartition par catégorie ===
+------------+-----+
|    category|count|
+------------+-----+
|       Autre|   39|
|Électronique|   23|
|Beauté/Santé|    6|
|  Musique/DJ|    4|
|      Livres|    2|
+------------+-----+



## 4. Exploration après nettoyage

In [9]:
print("=== Dépenses par année ===")
df.groupBy("order_year") \
  .agg(
      F.count("*").alias("nb_commandes"),
      F.round(F.sum("Total Amount"), 2).alias("total_depense_eur"),
      F.round(F.avg("Total Amount"), 2).alias("panier_moyen_eur")
  ) \
  .orderBy("order_year") \
  .show()

print("\n=== Total dépensé ===")
df.agg(F.round(F.sum("Total Amount"), 2).alias("total_eur")).show()

=== Dépenses par année ===
+----------+------------+-----------------+----------------+
|order_year|nb_commandes|total_depense_eur|panier_moyen_eur|
+----------+------------+-----------------+----------------+
|      NULL|           6|              0.0|             0.0|
|      2021|           5|           112.61|           22.52|
|      2022|          15|           342.02|            22.8|
|      2023|           6|            42.69|            7.12|
|      2024|          16|           877.97|           54.87|
|      2025|          13|           198.73|           15.29|
|      2026|          13|           123.88|            9.53|
+----------+------------+-----------------+----------------+


=== Total dépensé ===
+---------+
|total_eur|
+---------+
|   1697.9|
+---------+



In [10]:
print("=== Commandes par jour de la semaine ===")
df.groupBy("order_weekday") \
  .count() \
  .orderBy("order_weekday") \
  .show()

print("\n=== Commandes par heure ===")
df.groupBy("order_hour") \
  .count() \
  .orderBy("order_hour") \
  .show()

print("\n=== Top 10 produits les plus achetés ===")
df.groupBy("Product Name") \
  .count() \
  .orderBy(F.desc("count")) \
  .limit(10) \
  .show(truncate=60)

=== Commandes par jour de la semaine ===
+-------------+-----+
|order_weekday|count|
+-------------+-----+
|         NULL|    6|
|            1|   17|
|            2|    8|
|            3|    8|
|            4|   11|
|            5|    7|
|            6|    9|
|            7|    8|
+-------------+-----+


=== Commandes par heure ===
+----------+-----+
|order_hour|count|
+----------+-----+
|      NULL|    6|
|         5|    2|
|         6|    1|
|         8|    1|
|        10|    3|
|        11|    1|
|        12|    4|
|        13|    3|
|        14|    6|
|        15|    2|
|        16|    7|
|        17|    8|
|        18|   10|
|        19|    9|
|        20|    7|
|        21|    1|
|        22|    3|
+----------+-----+


=== Top 10 produits les plus achetés ===
+------------------------------------------------------------+-----+
|                                                Product Name|count|
+------------------------------------------------------------+-----+
|RISEOFLE 180cm 

In [11]:
print("=== Dépenses par catégorie ===")
df.groupBy("category") \
  .agg(
      F.count("*").alias("nb_achats"),
      F.round(F.sum("Total Amount"), 2).alias("total_eur"),
      F.round(F.avg("Total Amount"), 2).alias("panier_moyen")
  ) \
  .orderBy(F.desc("total_eur")) \
  .show()

print("\n=== Commandes avec remise ===")
df.filter(F.col("Total Discounts") < 0) \
  .agg(
      F.count("*").alias("nb_commandes_avec_remise"),
      F.round(F.sum("Total Discounts"), 2).alias("total_remises_eur")
  ).show()

=== Dépenses par catégorie ===
+------------+---------+---------+------------+
|    category|nb_achats|total_eur|panier_moyen|
+------------+---------+---------+------------+
|Électronique|       23|  1097.66|       47.72|
|       Autre|       39|   434.66|       11.15|
|Beauté/Santé|        6|    65.62|       10.94|
|  Musique/DJ|        4|     62.8|        15.7|
|      Livres|        2|    37.16|       18.58|
+------------+---------+---------+------------+


=== Commandes avec remise ===
+------------------------+-----------------+
|nb_commandes_avec_remise|total_remises_eur|
+------------------------+-----------------+
|                      10|           -20.67|
+------------------------+-----------------+



## 5. Schéma final & renommage

In [12]:
df_final = df.select(
    F.col("ASIN").alias("asin"),
    F.col("Order ID").alias("order_id"),
    F.col("Order Status").alias("order_status"),
    F.col("Product Name").alias("product_name"),
    F.col("category"),
    F.col("Original Quantity").alias("quantity"),
    F.col("Unit Price").alias("unit_price"),
    F.col("Total Amount").alias("total_amount"),
    F.col("Total Discounts").alias("total_discounts"),
    F.col("Shipping Charge").alias("shipping_charge"),
    F.col("Shipment Item Subtotal").alias("subtotal"),
    F.col("Payment Method Type").alias("payment_method"),
    F.col("Website").alias("website"),
    F.col("order_date"),
    F.col("order_year"),
    F.col("order_month"),
    F.col("order_weekday"),
    F.col("order_hour"),
    F.lit(3.0).alias("interaction_weight"),  # Achat = poids fort pour ALS
    F.lit("amazon").alias("platform")
)

print(f"Lignes finales : {df_final.count():,}")
df_final.printSchema()
df_final.show(5, truncate=50)

Lignes finales : 74
root
 |-- asin: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = false)
 |-- quantity: integer (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- total_discounts: double (nullable = true)
 |-- shipping_charge: double (nullable = true)
 |-- subtotal: double (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- website: string (nullable = true)
 |-- order_date: timestamp (nullable = true)
 |-- order_year: integer (nullable = true)
 |-- order_month: string (nullable = true)
 |-- order_weekday: integer (nullable = true)
 |-- order_hour: integer (nullable = true)
 |-- interaction_weight: double (nullable = false)
 |-- platform: string (nullable = false)

+----------+-------------------+------------+--------------------------------------------------+------------+--

## 6. Écriture Parquet

In [13]:
df_final.write \
    .mode("overwrite") \
    .parquet(PARQUET_OUT)

print(f"✓ Parquet écrit : {PARQUET_OUT}")

# Vérification
df_check = spark.read.parquet(PARQUET_OUT)
print(f"✓ Vérification lecture : {df_check.count():,} lignes")
df_check.show(3, truncate=50)

✓ Parquet écrit : ../../data/parquet/amazon_orders.parquet
✓ Vérification lecture : 74 lignes
+----------+-------------------+------------+--------------------------------------------------+------------+--------+----------+------------+---------------+---------------+--------+-----------------+-------------+-------------------+----------+-----------+-------------+----------+------------------+--------+
|      asin|           order_id|order_status|                                      product_name|    category|quantity|unit_price|total_amount|total_discounts|shipping_charge|subtotal|   payment_method|      website|         order_date|order_year|order_month|order_weekday|order_hour|interaction_weight|platform|
+----------+-------------------+------------+--------------------------------------------------+------------+--------+----------+------------+---------------+---------------+--------+-----------------+-------------+-------------------+----------+-----------+-------------+----------

In [14]:
spark.stop()